In [91]:
import numpy as np
from pysr import PySRRegressor

In [92]:
from sympy import Symbol, simplify, expand,symbols, integrate

In [ ]:
import pickle
import torch

# Load the .pkl file
with open('xy_8x8_data_with_score_obtained_through_score_model.pkl', 'rb') as file:
    data = pickle.load(file)

# Inspect the loaded data
print("Type of data:", type(data))

Type of data: <class 'list'>


In [94]:
data[0].shape

torch.Size([25000, 1, 32, 32])

## Fit PySR

In [ ]:
def extract_patches(field):
    """Extract patches with zero padding for boundary elements, selecting only center, up, down, left, and right."""
    # Pad the field with zeros
    padded_field = np.pad(field, pad_width=1, mode='wrap')
    patches = []
    grid_size = field.shape[0]

    for i in range(grid_size):
        for j in range(grid_size):
            # Extract the center and its immediate neighbors
            patch = np.array([
                padded_field[i + 1, j + 1],   # centre
                padded_field[i + 1, j + 2],      
                padded_field[i + 2, j + 1],  
                padded_field[i, j + 1],
                padded_field[i + 1, j],
               
            ])
            patches.append(patch)
    return np.array(patches)

def extract_patches_for_all(configurations):
    """Extract patches for all configurations."""
    all_patches = []
    for config in configurations:
        patches = extract_patches(config)
        all_patches.extend(patches)
    return np.array(all_patches)

grid_size = 8
num_conf = 400
mass = 1.0

patches_all = extract_patches_for_all((data[0][:num_conf,:,:,0]))
# Flatten 3x3 patches for input to PySR
X = patches_all
y = np.array(data[1][:num_conf]).flatten()

# Fit the PySR model
model = PySRRegressor(
    niterations=100,  # Number of search iterations
    binary_operators=["+", "-", "*"],
    unary_operators=["square","cube"],
    #progress=True,
    populations = 20,
    maxsize = 25,
    batching=True,
    batch_size=128,
    extra_sympy_mappings={
        "*": lambda x, y: (
            (1) * x * y if x == y  # Scale same-feature products by 1/3
            else (1 * x * y if isinstance(x, Symbol) and isinstance(y, Symbol)  # Scale feature-feature by 0.5
                  else x * y)  # Leave constant-feature products unchanged
        )
    },
    verbosity = 1,
    procs= 16,
)

In [96]:
X.shape

(204800, 4)

In [ ]:
# Train the model
model.fit(X,y)

In [ ]:
expand(simplify(model.sympy()))

## Compute MSE and R2

In [76]:
def mean_normalize(data):
    return (data - np.mean(data))

In [ ]:
import pickle
import torch

# Load the .pkl file
with open('xy_8x8_data.pkl', 'rb') as file:
    data_0 = pickle.load(file)
    if isinstance(data_0[0], torch.Tensor):
      variables = data_0[0].cpu().detach().numpy()  # Move to CPU and convert to NumPy array
    else:
      variables = data_0[0]
    if isinstance(data_0[1], torch.Tensor):
      targets_ = data_0[1].cpu().detach().numpy()  # Move to CPU and convert to NumPy array
    else:
      targets_ = data_0[1]
features = extract_patches_for_all(variables[:,0,:,:])
targets = (mean_normalize(targets_))
wn = 1

# Inspect the loaded data
print("Type of data:", type(data_0))

Type of data: <class 'list'>


In [ ]:
targets = (mean_normalize(targets_))


In [79]:
variables.shape

(25000, 1, 32, 32)

In [ ]:
x0, x1, x2, x3, x4, x5 = symbols('x0 x1 x2 x3 x4 x5')

In [83]:
from sympy import Add, expand

In [ ]:
#intrgrate using sympy
expr =  integrate(expand(model.equations_.sympy_format[len(model.equations_[0].sympy_format)-1]),x0) # lowest MSE equation
eq = expr#sum(new_terms)
expand(eq)

In [86]:
import sympy as sp
from sklearn.metrics import r2_score, mean_squared_error


In [ ]:
#calculate mse and r2
symbols_  = eq.free_symbols
# Given expression with fewer variables
x0, x1, x2, x3, x4, x5, x6, x7, x8, x9 = sp.symbols('x0 x1 x2 x3 x4 x5 x6 x7 x8 x9')  # Variables in your potential data
# Step 2: Create a mapping from symbol names to data indices
all_symbols = [x0, x1, x2, x3, x4, x5, x6, x7, x8, x9]  # Known order of variables
symbol_index_map = {symbol: i for i, symbol in enumerate(all_symbols)}
# Step 3: Subset data columns matching variables in the expression
relevant_indices = [symbol_index_map[var] for var in symbols_]
subset_data = features[:, relevant_indices]  # Select relevant columns

#globals().update({str(symbol): sp.symbols(str(symbol)) for symbol in symbols})
numerical_function = sp.lambdify(tuple(symbols_), eq, modules="numpy")
# Step 5: Evaluate the expression for each row
y = numerical_function(*subset_data.T)
y_prd =  mean_normalize(y)
# r2_score

mse = np.round(mean_squared_error(y_pred=mean_normalize(np.sum(np.array(y).reshape(-1,wn),axis=1)),y_true=targets),4)
r2 = np.round(r2_score(y_pred=np.sum(np.array(y_prd).reshape(-1,wn),axis=1),y_true=targets),4)
# with open('extra/results.txt', "a") as file:
#     file.write('multivariate_data_for_multi_mean_unit_cov_with_neg_score_through_nf_model_new.txt' + '\n')
#     file.write(str([(eq),f'mse: {mse}',f'r2: {r2}']) + '\n')
print("mse:",mse,"r2:",r2)

In [88]:
subset_data.shape

(25600000, 3)